# SAM-MedSeg: Inference Demo
---
**Few-Shot Medical Image Segmentation with Improved SAM**

This notebook demonstrates single-image inference using the trained SAM-MedSeg model.
It loads the best checkpoint (100% data, seed 44, Dice 0.652) and runs on a test image.

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Add project root to path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from models.full_model import SAMMedSeg
from data.dataset import MedicalSegDataset
from data.transforms import get_test_transforms
from utils.visualize import overlay_mask, plot_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ── Load the best SAM-MedSeg model ──
CHECKPOINT_PATH = "experiments/results/ratio_1.0_seed_44/best_model.pth"
SAM_CKPT = "checkpoints/sam_vit_b_01ec64.pth"

print("Loading SAM-MedSeg model...")
model = SAMMedSeg(
    sam_checkpoint=SAM_CKPT,
    model_type="vit_b",
    freeze_layers=9,
    lora_r=4,
    lora_alpha=16,
    lora_dropout=0.1,
    use_checkpoint=False,  # no gradient checkpointing for inference
    use_lora=True,
    device=device,
)

# Load trained weights
state_dict = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
model.load_state_dict(state_dict, strict=False)
model.eval()

# Merge LoRA for faster inference
model.merge_lora()

print(f"Model loaded. Fusion alpha: {model.get_fusion_alpha():.4f}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Load and preprocess a test image ──
DATA_DIR = PROJECT_ROOT / "data" / "kvasir-seg"

# Pick a test image (change index to try different images)
test_images = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))
test_masks = sorted((DATA_DIR / "test" / "masks").glob("*.jpg"))

idx = 5  # try indices 0–99
img_path = test_images[idx]
mask_path = test_masks[idx]

print(f"Image: {img_path.name}")
print(f"Mask:  {mask_path.name}")

# Load raw image for display
raw_img = Image.open(img_path).convert("RGB")
raw_mask = Image.open(mask_path).convert("L")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw_img)
axes[0].set_title("Input Image")
axes[0].axis("off")
axes[1].imshow(raw_mask, cmap="gray")
axes[1].set_title("Ground Truth Mask")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ── Run inference ──
from torchvision.transforms import functional as TF

# Preprocess: resize to 1024×1024 and normalize
img = raw_img.resize((1024, 1024), Image.BILINEAR)
img_tensor = TF.to_tensor(img).unsqueeze(0).to(device)  # (1, 3, 1024, 1024)

print(f"Input tensor shape: {img_tensor.shape}")
print("Running inference...")

with torch.no_grad():
    output = model(img_tensor, return_all=True)

final_mask = output["final_mask"]      # logits (1, 1, 1024, 1024)
sam_mask = output.get("sam_mask")      # SAM path output
med_mask = output.get("med_mask")      # MedDecoder output
fusion_alpha = output.get("fusion_alpha", 0.0)

# Apply sigmoid and threshold
final_prob = torch.sigmoid(final_mask).squeeze().cpu().numpy()
final_binary = (final_prob > 0.5).astype(np.float32)

print(f"Fusion alpha: {fusion_alpha:.4f}")
print(f"Predicted foreground ratio: {final_binary.mean():.4f}")

In [ ]:
# ── Visualize results ──
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

# Input image
axes[0].imshow(img)
axes[0].set_title("Input Image")
axes[0].axis("off")

# Ground truth (resized to match)
gt = raw_mask.resize((1024, 1024), Image.NEAREST)
axes[1].imshow(gt, cmap="gray")
axes[1].set_title("Ground Truth")
axes[1].axis("off")

# Final prediction (SAM-MedSeg)
axes[2].imshow(final_binary, cmap="gray")
axes[2].set_title(f"SAM-MedSeg\nDice={final_prob.mean():.3f}")
axes[2].axis("off")

# Probability heatmap
im = axes[3].imshow(final_prob, cmap="hot", vmin=0, vmax=1)
axes[3].set_title("Probability Map")
axes[3].axis("off")
plt.colorbar(im, ax=axes[3], fraction=0.046)

# Overlay
overlay = np.array(img).copy()
overlay[final_binary > 0.5] = overlay[final_binary > 0.5] * 0.5 + np.array([255, 0, 0]) * 0.5
axes[4].imshow(overlay)
axes[4].set_title("Overlay (Red=Pred)")
axes[4].axis("off")

plt.tight_layout()
plt.savefig("experiments/figures/demo_inference_example.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Compare SAM path vs MedDecoder path ──
if sam_mask is not None and med_mask is not None:
    sam_prob = torch.sigmoid(sam_mask).squeeze().cpu().numpy()
    med_prob = torch.sigmoid(med_mask).squeeze().cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(sam_prob, cmap="hot", vmin=0, vmax=1)
    axes[0].set_title(f"SAM Path (weight={fusion_alpha:.3f})")
    axes[0].axis("off")
    axes[1].imshow(med_prob, cmap="hot", vmin=0, vmax=1)
    axes[1].set_title(f"MedDecoder Path (weight={1-fusion_alpha:.3f})")
    axes[1].axis("off")
    axes[2].imshow(final_prob, cmap="hot", vmin=0, vmax=1)
    axes[2].set_title(f"Fused Output")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

## Summary

This demo shows SAM-MedSeg performing inference on a single test image.

**Key observations:**
- The SAM path (grid-sampled prompts) provides a strong semantic prior
- The MedDecoder path recovers fine boundary details via skip connections
- The learnable fusion (α ≈ 0.623) balances both sources

**Try it yourself:**
- Change `idx` in the "Load Image" cell to test different images (0–99)
- Compare with U-Net predictions by loading the U-Net checkpoint
- Experiment with different prompt strategies by modifying `PromptOptimizer` parameters